In [1]:
%cd ~/Documents/GitHub/MMMM

/Users/matthewziyusu/Documents/GitHub/MMMM


In [2]:
import sys

sys.path.append("./models")

In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from eeg_encoder import TextHead, EEGEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F

class TextHead(nn.Module):
    def __init__(self, input_dim=840, output_dim=1024, hidden_dim=1024, num_layers=2):
        super(TextHead, self).__init__()
        self.hidden_layers = nn.ModuleList()
        self.hidden_layers.append(nn.Linear(input_dim, hidden_dim))
        for i in range(num_layers - 1):
            self.hidden_layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.output_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = nn.ReLU()

    def forward(self, x):
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.activation(self.output_layer(x))
        return x

class EEGEncoder(nn.Module):
    def __init__(self, txt_in_feat=840, img_in_feat=500, enc_feat=1024, dec_emb_sz=768, enc_nhead=8, enc_dim_ff=2048, num_enc_layers=8):
        super(EEGEncoder, self).__init__()

        self.txt_head = TextHead(input_dim=txt_in_feat, output_dim=enc_feat)
        self.img_head = nn.Linear(img_in_feat, enc_feat)

        self.encoder_layer = nn.TransformerEncoderLayer(d_model=enc_feat, nhead=enc_nhead, dim_feedforward=enc_dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_enc_layers)
        self.fc_proj = nn.Linear(enc_feat, dec_emb_sz)

    def forward(self, mode, input_data_batch=None, input_masks_batch=None, input_masks_invert=None, avg_pool = False):
        if mode == "TXT":
            encoded_embedding = self.txt_head(input_data_batch)
        elif mode == "IMG":
            encoded_embedding = self.img_head(input_data_batch)

        encoded_embedding = self.encoder(input_data_batch, src_key_padding_mask = input_masks_invert)
        encoded_embedding = F.relu(self.fc_proj(encoded_embedding))
        
        if not avg_pool:
            return encoded_embedding
        else:
            sz = encoded_embedding.shape
            avg_embedding = torch.zeros(sz[:-1])
            for i in range(encoded_embedding.shape[-1]):
                avg_embedding += encoded_embedding[:][:][i]
            avg_embedding/=encoded_embedding.shape[-1]
            return avg_embedding

In [65]:
eeg_enc = EEGEncoder(txt_in_feat=10, enc_feat=10, dec_emb_sz=16, enc_nhead = 1, enc_dim_ff=16, num_enc_layers=8)
results = eeg_enc(mode="TXT", input_data_batch = test_data, input_masks_batch = masks, input_masks_invert = masks, avg_pool=False)

In [66]:
res = results

In [68]:
sz = res.shape; sz

torch.Size([8, 10, 16])

In [74]:
avg_embedding = torch.zeros(sz[:-2] + tuple([sz[-1]]))

In [75]:
avg_embedding.shape

torch.Size([8, 16])

In [78]:
for i in range(sz[-2]):
    avg_embedding += results[:,i]

In [80]:
avg_embedding /= sz[-2]

In [53]:
test_data = torch.randn(8, 10, 10)
# Batches, Batch Size, Data Dimension

In [42]:
text_hd = TextHead(10, 16)

In [43]:
with torch.no_grad():
    results = text_hd(test_data)
print(results)

tensor([[[1.7081e-01, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          8.9882e-03, 1.0313e-01],
         [9.3226e-03, 0.0000e+00, 1.1282e-02,  ..., 4.9563e-04,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         ...,
         [6.5352e-03, 0.0000e+00, 2.1021e-02,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [2.8361e-02, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 5.7171e-02],
         [6.2100e-02, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00]],

        [[1.6639e-01, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 4.7163e-02],
         [1.1372e-01, 1.5542e-01, 1.0028e-04,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [6.6032e-02, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         ...,
         [4.7563e-02, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.000

In [44]:
results.shape

torch.Size([8, 10, 16])

In [45]:
eeg_enc = EEGEncoder(txt_in_feat=10, enc_feat=10, dec_emb_sz=16, enc_nhead = 1, enc_dim_ff=16, num_enc_layers=8)

In [46]:
# masks = torch.ones(10)
# masks[0] = masks[1] = masks[2] = masks[3] = masks[4] = 0
masks = torch.zeros(8, 10)
inv_masks = torch.ones(8, 10)

for i in range(0, 8):
    masks[i][0] = masks[i][1] = masks[i][2] = masks[i][3] = masks[i][4] = 1
    inv_masks[i][0] = inv_masks[i][1] = inv_masks[i][2] = inv_masks[i][3] = inv_masks[i][4] = 0

In [47]:
results = eeg_enc(mode="TXT", input_data_batch = test_data, input_masks_batch = masks, input_masks_invert = masks, avg_pool=True)

IndexError: index 8 is out of bounds for dimension 0 with size 8

In [38]:
print(results)

tensor([[[0.0000, 0.0000, 0.8157,  ..., 0.3986, 0.0000, 0.3200],
         [0.0000, 0.4009, 0.8016,  ..., 0.0000, 0.0000, 0.4015],
         [0.9625, 0.6553, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0000, 0.4248, 0.4181,  ..., 0.0000, 0.3206, 0.1834],
         [0.0000, 0.7230, 0.9562,  ..., 0.0189, 0.0579, 0.3007],
         [0.0000, 0.3076, 1.1475,  ..., 0.4092, 0.2186, 0.2919]],

        [[0.0000, 0.4217, 0.2121,  ..., 0.0000, 0.5513, 0.2099],
         [0.3377, 0.0299, 0.0000,  ..., 0.0000, 0.1025, 0.0000],
         [0.0000, 0.1660, 1.1740,  ..., 0.2580, 0.2220, 0.6512],
         ...,
         [0.6367, 0.0000, 0.0000,  ..., 0.0000, 0.0488, 0.0000],
         [0.2302, 0.0000, 0.0000,  ..., 0.0000, 0.2658, 0.3633],
         [0.6802, 0.0000, 0.0000,  ..., 0.0000, 0.2606, 0.0000]],

        [[0.5693, 0.0000, 0.0000,  ..., 0.0000, 0.4263, 0.0000],
         [0.6800, 0.0000, 0.0000,  ..., 0.0000, 0.4855, 0.0000],
         [0.0000, 0.0000, 0.6877,  ..., 0.1188, 0.5524, 0.

In [21]:
results.shape

torch.Size([8, 10, 16])